1. Compared to a free notebook, there was a lot more setup before any code could run. I had to redeem the education credit, make sure the billing account was linked to the project, enable the Vertex AI API, and then create a runtime and pick a machine type and region. In a free Colab you just open the notebook and run. The biggest difference is the cost, the runtime bills the whole time it’s active, even when no cell is running. The notebook also lives inside the GCP project, so I had to download the .ipynb and upload it to GitHub myself, and take screenshots of the runtime and billing page as proof it ran on GCP.

2. info() showed 9,357 non-null values in every column, so it looked like there was no missing data. describe() showed that every column had a min of -200 and the means were not possible. -200 is the sentinel for a missing reading, so I replaced it with NaN. NMHC(GT) was about 90% missing its 25th, 50th, and 75th percentiles were all -200, so I dropped it instead of imputing mostly made up values. I dropped the 1,683 rows where the target CO(GT) was missing, since you can’t train on a label you invented, which left 7,674 rows. The remaining missing sensor values were filled in with the median using SimpleImputer. I picked the median because the gas columns are right skewed. Once the -200s were gone and the NaNs were imputed, the models trained without any problems. Logistic regression converged in 76 iterations and Ridge’s error stayed flat across.

3. If you impute or scale the whole dataset before splitting, the test set’s statistics leak into training. Putting the imputer and scaler inside a Pipeline and calling fit_transform only on X_train means the medians, means, and standard deviations are learned from training data only, and the test set just gets transform. The ColumnTransformer sits on top and sends the numerical columns through median imputation and StandardScaler, and hour, month, and dayofweek through most-frequent imputation and one-hot encoding, all in one step. That way I don’t have to track and reapply each transformation by hand. The -200 to NaN replacement happened before the split, but it’s a fixed rule and doesn’t learn anything from the data, so it doesn’t leak.

4. StandardScaler centers each feature at mean 0 with unit variance, and keeps the coefficients comparable. MinMaxScaler would squeeze each feature into a fixed range instead. In theory, it is more sensitive to outliers, since one extreme reading stretches the whole range and compresses the rest of the values. StandardScaler is also affected by outliers, but its output isn’t bounded. Since the gas and sensor columns are right skewed and have spikes, the two scalers would give similar results with possibly small differences.

5. LinearRegression and Ridge can be solved in closed form, but logistic regression has no closed-form solution, so LogisticRegression uses an iterative full-batch solver that sees all the data on every step. SGDClassifier instead updates the weights a little at a time using one sample per update, so it has extra settings to tune like the loss, learning rate, and alpha. It’s also faster on large data and supports incremental learning. In my results, logistic regression got 87.2% in 76 iterations and 1.4 s, while SGD with log_loss got 86.3% in 35 epochs and 0.16 s. The probability outputs from log_loss matter because a prediction of High with 95% confidence is different from one at 40%, and borderline readings near a threshold can be flagged. Hinge loss doesn’t give probabilities. The classification report and confusion matrix show more moderate was the hardest class, and almost all errors were between neighboring classes, only 4 of 1,535 rows jumping from Low to High or the reverse.

6. Ridge didnt change much as alpha increased. RMSE stayed at 0.4268 and R^2 at 0.913 from alpha 0.001 up to about 3, because with about 6,100 rows and standardized features there isn’t much variance to shrink. Lasso did feature selection at alpha 0.001 it zeroed 11 of 54 coefficients with no loss in accuracy, at alpha 0.1 only 4 features were left, and at alpha above about 1.5 every coefficient was zero and the model just predicted the mean. The strongest Lasso coefficients were PT08.S4, C6H6, and NOx, so CO tracks closely with the other pollutants. Hours 18 to 21 had positive coefficients and hour 3 was negative, and December was positive. So monitoring and alerts should focus on evenings and colder months. Since four features recovered most of the accuracy, a cheaper station could use a few sensors instead of the full set. PT08.S2 had a negative coefficient.